In [ ]:
# =============================================================================
# BOOTSTRAP: instalacion automatica de dependencias (self-contained, Linux-ready)
# =============================================================================
import importlib
import subprocess
import sys


def _bootstrap_dependencias(paquetes):
    faltantes = []
    for modulo, paquete_pip in paquetes.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltantes.append(paquete_pip)
    if faltantes:
        print(f"[INFO] Instalando dependencias faltantes: {', '.join(faltantes)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])


_bootstrap_dependencias({
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlalchemy": "sqlalchemy",
    "pyodbc": "pyodbc",
})

# hasattr(...): bajo Jupyter/ipykernel, sys.stdout/stderr son un OutStream
# propio que NO tiene .reconfigure() (solo lo tiene un TextIOWrapper real,
# el caso de un script corrido por CPython en terminal/cron/systemd) -- sin
# este chequeo, la celda revienta con AttributeError apenas se corre como
# notebook en un kernel.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

import os
import hashlib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, bindparam
from urllib.parse import quote_plus
import datetime


In [ ]:
# =============================================================================
# CONFIGURACION (autocontenido: sin dependencias de otro archivo)
# =============================================================================

def _verificar_driver_odbc(nombre_driver="ODBC Driver 17 for SQL Server"):
    """
    El driver ODBC de SQL Server es un paquete de sistema operativo, no
    instalable por pip. Se valida su presencia antes de conectar para fallar
    con un mensaje claro en vez de un traceback criptico de pyodbc.
    """
    import pyodbc
    disponibles = pyodbc.drivers()
    if nombre_driver not in disponibles:
        raise RuntimeError(
            f"No se encontro el driver ODBC '{nombre_driver}' en este servidor.\n"
            f"Drivers disponibles: {disponibles or '(ninguno)'}\n"
            "Instalar en Debian/Ubuntu:\n"
            "  curl -sSL https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -\n"
            "  curl -sSL https://packages.microsoft.com/config/ubuntu/22.04/prod.list | sudo tee /etc/apt/sources.list.d/mssql-release.list\n"
            "  sudo apt-get update && ACCEPT_EULA=Y sudo apt-get install -y msodbcsql17 unixodbc-dev"
        )


_verificar_driver_odbc()

conn_str_work = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27,1433;"
    "DATABASE=SANDBOX_BRONZE;"
    "UID=sa;"
    "PWD=M4rfr1g.2023*;"
    "TrustServerCertificate=yes;"
)
# fast_executemany=True: pyodbc manda cada lote de INSERT en un solo viaje al
# servidor en vez de una ida y vuelta por fila.
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_work}", fast_executemany=True)

conn_str_mds = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27\\mfgbi;"
    "DATABASE=MDS_PRODU;"
    "UID=pbi_report;"
    "PWD=Marfrig2024;"
    "TrustServerCertificate=yes;"
)
engine_mds = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_mds}")


In [ ]:
# =============================================================================
# CARGA INCREMENTAL POR SEMANA Y FUENTE
# -----------------------------------------------------------------------------
# Antes: cada corrida leia las dos silver completas y reemplazaba
# PRC.Gold_Comparativo entera (DROP + CREATE + INSERT en chunks de 1000).
#
# Gold no calcula nada entre semanas: apila COPI y SNP y les pega maestros
# por fila. La unidad de trabajo es (Fuente, Fecha):
#   1. Cada silver deja en PRC.silver_scanntech_semanas / PRC.SNP_silver_semanas
#      que lunes escribio y cuando (fecha_proceso, GETDATE() del mismo SQL
#      Server: sin problemas de relojes).
#   2. Gold guarda en PRC.Gold_Comparativo_semanas la version (fecha_proceso
#      de silver) que proceso de cada (Fuente, Fecha). Todo lunes nuevo,
#      rehecho o eliminado en silver queda pendiente.
#   3. DELETE de (Fuente, Fecha) afectados + INSERT + control en UNA transaccion.
#
# Full rebuild: HORUS_FULL_REBUILD=1, cambio en maestros MDS (EANs, PDVs
# Scanntech, Bocas SNP: afectan a toda la historia), cambio de VERSION_LOGICA,
# primera corrida, tabla inexistente, o silver sin tablas de control (silver
# todavia en version no incremental).
# HORUS_DRY_RUN=1: muestra el plan sin escribir nada.
# =============================================================================

SCHEMA          = 'PRC'
TABLA_GOLD      = 'Gold_Comparativo'
TABLA_SEMANAS   = 'Gold_Comparativo_semanas'
TABLA_CONTROL   = 'horus_control'
CLAVE_MAESTROS  = 'gold_scanntech_snp.maestros'
VERSION_LOGICA  = '2026-09-11'
CHUNK_SIZE      = 20_000

FUENTES = {
    # Fuente: (tabla silver, tabla de control por semana que deja silver)
    'COPI': ('silver_scanntech', 'silver_scanntech_semanas'),
    'SNP':  ('SNP_silver',       'SNP_silver_semanas'),
}


def _flag_env(nombre):
    return os.environ.get(nombre, '').strip().lower() in ('1', 'true', 'yes', 'si')


FULL_REBUILD_ENV = _flag_env('HORUS_FULL_REBUILD')
DRY_RUN          = _flag_env('HORUS_DRY_RUN')


def fingerprint_maestros(*dfs):
    h = hashlib.sha256(VERSION_LOGICA.encode())
    for df in dfs:
        d = df.astype(str)
        d = d.sort_values(list(d.columns)).reset_index(drop=True)
        h.update(pd.util.hash_pandas_object(d, index=False).values.tobytes())
    return h.hexdigest()


def _in_clause(sql, nombre):
    return text(sql).bindparams(bindparam(nombre, expanding=True))


def tabla_existe(conn, tabla):
    return conn.execute(text("SELECT OBJECT_ID(:n, 'U')"), {'n': f'{SCHEMA}.{tabla}'}).scalar() is not None


def asegurar_tablas_control(conn):
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_SEMANAS}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_SEMANAS}] (
            Fuente               NVARCHAR(10) NOT NULL,
            Fecha                DATE NOT NULL,
            fecha_proceso_silver DATETIME NULL,
            filas                INT,
            fecha_proceso        DATETIME DEFAULT GETDATE(),
            PRIMARY KEY (Fuente, Fecha)
        )
    """))
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_CONTROL}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_CONTROL}] (
            clave NVARCHAR(200) PRIMARY KEY,
            valor NVARCHAR(MAX),
            fecha DATETIME DEFAULT GETDATE()
        )
    """))


def leer_control(conn, clave):
    return conn.execute(text(f"SELECT valor FROM [{SCHEMA}].[{TABLA_CONTROL}] WHERE clave = :k"),
                        {'k': clave}).scalar()


def escribir_control(conn, clave, valor):
    conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_CONTROL}] WHERE clave = :k"), {'k': clave})
    conn.execute(text(f"INSERT INTO [{SCHEMA}].[{TABLA_CONTROL}] (clave, valor) VALUES (:k, :v)"),
                 {'k': clave, 'v': valor})


def leer_semanas_silver(conn):
    """
    {Fuente: {Fecha: fecha_proceso}} segun las tablas de control de silver.
    None si alguna silver no tiene tabla de control (version no incremental):
    en ese caso gold no puede saber que cambio y rehace todo.
    """
    estado = {}
    for fuente, (_, tabla_ctl) in FUENTES.items():
        if not tabla_existe(conn, tabla_ctl):
            print(f"[WARN] No existe [{SCHEMA}].[{tabla_ctl}]: la silver de {fuente} aun no corre "
                  "en modo incremental. Gold rehace todo.")
            return None
        rows = conn.execute(text(f"SELECT Fecha, fecha_proceso FROM [{SCHEMA}].[{tabla_ctl}]")).fetchall()
        estado[fuente] = {r[0]: r[1] for r in rows}
    return estado


def leer_semanas_gold(conn):
    """{Fuente: {Fecha: fecha_proceso_silver}} procesado por gold."""
    rows = conn.execute(text(
        f"SELECT Fuente, Fecha, fecha_proceso_silver FROM [{SCHEMA}].[{TABLA_SEMANAS}]"
    )).fetchall()
    estado = {f: {} for f in FUENTES}
    for r in rows:
        estado.setdefault(r[0], {})[r[1]] = r[2]
    return estado


def determinar_pendientes(silver, gold):
    """{Fuente: (fechas_rehacer, fechas_eliminar)} comparando estado silver vs gold."""
    plan = {}
    for fuente in FUENTES:
        sv, gd = silver.get(fuente, {}), gold.get(fuente, {})
        rehacer  = sorted(f for f, fp in sv.items() if gd.get(f, object()) != fp)
        eliminar = sorted(f for f in gd if f not in sv)
        plan[fuente] = (rehacer, eliminar)
    return plan


def _a_datetime(fechas):
    return [datetime.datetime.combine(f, datetime.time()) for f in fechas]


def _sql_filtro_fechas(fechas):
    """(clausula WHERE, params) para filtrar por Fecha; sin filtro si fechas es None."""
    if fechas is None:
        return '', {}
    return ' WHERE Fecha IN :fechas', {'fechas': _a_datetime(fechas)}


def _read_sql_fechas(sql, fechas):
    """Lee sql (sin WHERE) filtrando por Fecha en lotes de 500 fechas, o completo si fechas is None."""
    if fechas is None:
        return pd.read_sql(sql, engine)
    if not fechas:
        return pd.read_sql(sql.replace('SELECT', 'SELECT TOP 0', 1), engine)
    partes = []
    with engine.connect() as conn:
        for i in range(0, len(fechas), 500):
            where, params = _sql_filtro_fechas(fechas[i:i + 500])
            partes.append(pd.read_sql(_in_clause(sql + where, 'fechas'), conn, params=params))
    return pd.concat(partes, ignore_index=True)


def borrar_fuente_fechas(conn, fuente, fechas, tabla=None, columna_fecha='Fecha'):
    tabla = tabla or TABLA_GOLD
    if not fechas:
        return
    for i in range(0, len(fechas), 500):
        stmt = _in_clause(
            f"DELETE FROM [{SCHEMA}].[{tabla}] WHERE Fuente = :fuente AND {columna_fecha} IN :fechas", 'fechas')
        conn.execute(stmt, {'fuente': fuente, 'fechas': fechas[i:i + 500]})


def crear_tabla_desde_df(conn, df, tabla):
    """
    Crea (o recrea) la tabla con el esquema inferido del DataFrame COMPLETO y
    un indice clustered por Fecha (acelera el DELETE por semana del incremental).
    Solo Fecha: Fuente queda como NVARCHAR(MAX) y no puede ser clave de indice.
    """
    df.head(0).to_sql(tabla, con=conn, schema=SCHEMA, if_exists='replace', index=False)
    conn.execute(text(f"CREATE CLUSTERED INDEX [IX_{tabla}_Fecha] ON [{SCHEMA}].[{tabla}] (Fecha)"))


def insertar(conn, df, tabla):
    total = len(df)
    print(f"[INFO] Insertando {total:,} filas en {SCHEMA}.{tabla}...")
    if total:
        df.to_sql(tabla, con=conn, schema=SCHEMA, if_exists='append', index=False, chunksize=CHUNK_SIZE)
    print(f"[OK] {SCHEMA}.{tabla}: {total:,} filas.")


In [ ]:
# =============================================================================
# FUNCIONES
# =============================================================================

def leer_silver_copi(fechas=None):
    """
    Lee la tabla silver de Scanntech (COPI) y selecciona las columnas relevantes.
    Incluye KG (volumen en kilogramos) calculado en la capa silver.
    Excluye Facturacion y Cantidad (no comparables con SNP).
    fechas=None lee todo; una lista de lunes lee solo esas semanas.
    """
    print("[INFO] Leyendo PRC.silver_scanntech (COPI)...")
    df = _read_sql_fechas("""
        SELECT EAN, Fecha, Codigo_PDV, Precio_Dinamizado, Precio_BASE, KG
        FROM [PRC].[silver_scanntech]
    """.strip(), fechas)
    print(f"   {len(df):,} filas leidas.")

    df['EAN']        = df['EAN'].astype(str).str.strip()
    df['Fecha']      = pd.to_datetime(df['Fecha'], errors='coerce')
    df['Codigo_PDV'] = df['Codigo_PDV'].astype(str).str.strip()

    df_copi = pd.DataFrame({
        'EAN':               df['EAN'],
        'Fecha':             df['Fecha'],
        'Codigo_PDV':        df['Codigo_PDV'],
        'Precio':            pd.to_numeric(df.get('Precio_Dinamizado'), errors='coerce'),
        'Precio_BASE':       pd.to_numeric(df.get('Precio_BASE'),       errors='coerce'),
        'Precio_Dinamizado': pd.to_numeric(df.get('Precio_Dinamizado'), errors='coerce'),
        'KG':                pd.to_numeric(df.get('KG'),                errors='coerce'),
        'Fuente':            'COPI'
    })

    return df_copi


def leer_silver_snp(fechas=None):
    """
    Lee la tabla silver de SNP y selecciona las columnas relevantes.
    Usa EAN_Final (EAN canonico resuelto desde el maestro) como clave de join.
    Incluye Supervisor (disponible en el formato nuevo de relevamiento).
    fechas=None lee todo; una lista de lunes lee solo esas semanas.
    """
    print("[INFO] Leyendo PRC.SNP_silver (SNP)...")

    cols_snp = pd.read_sql("SELECT TOP 0 * FROM [PRC].[SNP_silver]", engine).columns.tolist()
    ean_col        = 'EAN_Final' if 'EAN_Final' in cols_snp else 'EAN'
    supervisor_sql = ', Supervisor' if 'Supervisor' in cols_snp else ''
    cadena_sql     = ', Cadena' if 'Cadena' in cols_snp else ''

    df = _read_sql_fechas(f"""
        SELECT {ean_col} AS EAN, Fecha, Codigo_PDV,
               Precio_Dinamizado, Precio_BASE{supervisor_sql}{cadena_sql}
        FROM [PRC].[SNP_silver]
    """.strip(), fechas)
    print(f"   {len(df):,} filas leidas.")

    df['EAN']        = df['EAN'].astype(str).str.strip()
    df['Fecha']      = pd.to_datetime(df['Fecha'], errors='coerce')
    df['Codigo_PDV'] = df['Codigo_PDV'].astype(str).str.strip()

    df_snp = pd.DataFrame({
        'EAN':               df['EAN'],
        'Fecha':             df['Fecha'],
        'Codigo_PDV':        df['Codigo_PDV'],
        'Precio':            pd.to_numeric(df.get('Precio_Dinamizado'), errors='coerce'),
        'Precio_BASE':       pd.to_numeric(df.get('Precio_BASE'),       errors='coerce'),
        'Precio_Dinamizado': pd.to_numeric(df.get('Precio_Dinamizado'), errors='coerce'),
        'Supervisor':        df.get('Supervisor'),
        'Cadena_SNP':        df.get('Cadena'),
        'Fuente':            'SNP'
    })

    return df_snp


def leer_maestro_eans():
    print("\n[INFO] Cargando dimensiones de producto desde EAN_Mercado...")
    df_eans = pd.read_sql("""
        SELECT
            EAN,
            Familia, Empresa, Marca, Segmento,
            [Cód SKU Marfrig]       AS Cod_SKU,
            [Descripción SKU Nuevo] AS Descripcion_SKU,
            [Marca Propia_Name]     AS Marca_Propia
        FROM [mdm].[ICO_EANs_mercado]
    """, engine_mds)
    df_eans['EAN'] = df_eans['EAN'].astype(str).str.strip()
    return df_eans


def construir_maestro_pdv():
    """
    Construye el maestro combinado de PDVs haciendo append de ambas fuentes.

    COPI - [mdm].[ICO_PDVs_Scanntech]:
        Codigo_PDV (Scanntech_<id>), Nombre_PDV, Direccion_PDV,
        Provincia, Region, Localidad, Cluster_Scanntech, Cluster_Para, Subcanal, CUIT

    SNP - [mdm].[ICO_SNP_Bocas] (vista de suscripcion MDS, gemela de la de
    COPI -- stg.[SNP Bocas_Leaf] es solo el staging interno de MDS, vacio por
    diseno; los miembros reales se exponen por esta vista):
        Codigo_PDV (SNP_<Cod boca>), Nombre_PDV (Descripcion Boca),
        Provincia, Region, Subcanal (Canal IC),
        Cod_Cliente (Cod Cli Marfrig), Descripcion_Cliente,
        Cod_Boca (Cod boca), Cod_Boca_Marfrig (Cod Boca Marfrig)

    COPI va primero: drop_duplicates(keep='first') prioriza COPI en caso de
    colision por Codigo_PDV (en la practica no ocurre por diferencia de prefijo).
    Campos ausentes en cada fuente quedan como NaN.
    """
    print("[INFO] Construyendo maestro PDVs combinado...")

    df_copi = pd.read_sql("""
        SELECT DISTINCT
            [Código único PDV]  AS _codigo_raw,
            [Nombre PDV]        AS Nombre_PDV,
            [Dirección PDV]     AS Direccion_PDV,
            QF_Provincia        AS Provincia,
            Región              AS Region,
            Localidad,
            [Clúster Scanntech] AS Cluster_Scanntech,
            [Clúster Para]      AS Cluster_Para,
            [Subcanal Para]     AS Subcanal,
            CUIT
        FROM [mdm].[ICO_PDVs_Scanntech]
    """, engine_mds)
    df_copi['Codigo_PDV'] = 'Scanntech_' + df_copi['_codigo_raw'].astype(str).str.strip()
    df_copi = df_copi.drop(columns=['_codigo_raw'])
    df_copi['Fuente_PDV'] = 'COPI'

    df_snp = pd.read_sql("""
        SELECT DISTINCT
            [Cód boca]            AS Cod_Boca,
            [Descripción Boca]    AS Nombre_PDV,
            [Cód Cli Marfrig]     AS Cod_Cliente,
            [Descripción Cliente] AS Descripcion_Cliente,
            [Cód Boca Marfrig]    AS Cod_Boca_Marfrig,
            [Canal IC]            AS Subcanal,
            Provincia,
            Región                AS Region
        FROM [mdm].[ICO_SNP_Bocas]
    """, engine_mds)
    df_snp['Cod_Boca']   = df_snp['Cod_Boca'].astype(str).str.strip()
    df_snp['Codigo_PDV'] = 'SNP_' + df_snp['Cod_Boca']
    if 'Region' in df_snp.columns:
        df_snp['Region'] = df_snp['Region'].astype(str).str.strip().replace({'NEa': 'NEA'})
    df_snp['Fuente_PDV'] = 'SNP'

    df_pdv = pd.concat([df_copi, df_snp], ignore_index=True).drop_duplicates(
        subset=['Codigo_PDV'], keep='first'
    )
    print(f"   PDVs COPI: {len(df_copi):,} | PDVs SNP: {len(df_snp):,} | Total unificado: {len(df_pdv):,}")
    return df_pdv


def construir_tabla_comparativa(df_copi, df_snp, df_eans, df_pdv):
    """
    Apila COPI y SNP en una tabla unificada (concat). Los Codigo_PDV llevan
    prefijos distintos por fuente ('Scanntech_' / 'SNP_'), por lo que nunca
    puede haber coincidencias cruzadas entre fuentes: un outer join por
    EAN+Fecha+Codigo_PDV es equivalente a este concat en el resultado, pero
    mas costoso (hace un hash-join completo que nunca encuentra matches) y
    obliga a resolver columnas duplicadas con sufijos _copi/_snp.

    Las dimensiones de producto se toman del maestro EAN_Mercado.
    Las dimensiones de PDV se incorporan desde el maestro combinado de PDVs.
    """
    df_eans = df_eans.drop_duplicates(subset='EAN', keep='first')

    print("\n[INFO] Construyendo tabla comparativa...")
    df_copi['EAN'] = df_copi['EAN'].astype(str)
    df_snp['EAN']  = df_snp['EAN'].astype(str)

    df = pd.concat([df_copi, df_snp], ignore_index=True)

    # En_COPI/En_SNP se mantienen por compatibilidad con consumidores existentes
    df['En_COPI'] = df['Fuente'] == 'COPI'
    df['En_SNP']  = df['Fuente'] == 'SNP'

    # Dimensiones de producto desde EAN_Mercado (deduplicado)
    df = pd.merge(df, df_eans, on='EAN', how='left')

    # Dimensiones de PDV desde maestro combinado (COPI priorizado)
    df = pd.merge(df, df_pdv, on='Codigo_PDV', how='left')

    # Fallback para bocas SNP que aun no figuran en el maestro de PDVs:
    # Descripcion_Cliente se completa con la Cadena derivada en silver
    # (prefijo de Descripcion_Boca). Solo rellena donde el maestro no aporto.
    if 'Cadena_SNP' in df.columns:
        if 'Descripcion_Cliente' not in df.columns:
            df['Descripcion_Cliente'] = pd.NA
        df['Descripcion_Cliente'] = df['Descripcion_Cliente'].fillna(df['Cadena_SNP'])

    columnas = [
        # Producto
        'EAN', 'Cod_SKU', 'Descripcion_SKU', 'Familia', 'Empresa',
        'Marca', 'Segmento', 'Marca_Propia',
        # Tiempo
        'Fecha',
        # PDV - campos compartidos COPI + SNP
        'Codigo_PDV', 'Nombre_PDV', 'Provincia', 'Region', 'Subcanal',
        # PDV - campos exclusivos COPI
        'Direccion_PDV', 'Localidad', 'Cluster_Scanntech', 'Cluster_Para', 'CUIT',
        # PDV - campos exclusivos SNP
        'Cod_Boca', 'Cod_Boca_Marfrig', 'Cod_Cliente', 'Descripcion_Cliente',
        # Metricas
        'Precio', 'Precio_BASE', 'Precio_Dinamizado', 'KG',
        # Trazabilidad
        'Fuente', 'Fuente_PDV', 'Supervisor', 'En_COPI', 'En_SNP',
    ]
    df = df[[c for c in columnas if c in df.columns]]

    return df


def imprimir_resumen(df):
    total     = len(df)
    solo_copi = int(df['En_COPI'].sum())
    solo_snp  = int(df['En_SNP'].sum())

    print("\n[INFO] RESUMEN TABLA GOLD:")
    print(f"   Total registros:    {total:,}")
    if total > 0:
        print(f"   Registros COPI:     {solo_copi:,} ({solo_copi/total*100:.1f}%)")
        print(f"   Registros SNP:      {solo_snp:,} ({solo_snp/total*100:.1f}%)")
    else:
        print("   Sin datos.")


In [ ]:
# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def run_gold_comparativo():
    print(f"--- INICIO PROCESO GOLD COMPARATIVO: {datetime.datetime.now()} ---\n")

    with engine.begin() as conn:
        asegurar_tablas_control(conn)

    # 1. MAESTROS + DECISION FULL / INCREMENTAL
    df_eans   = leer_maestro_eans()
    df_pdv    = construir_maestro_pdv()
    fp_actual = fingerprint_maestros(df_eans, df_pdv)

    with engine.connect() as conn:
        fp_previo   = leer_control(conn, CLAVE_MAESTROS)
        estado_sv   = leer_semanas_silver(conn)
        estado_gold = leer_semanas_gold(conn)
        gold_existe = tabla_existe(conn, TABLA_GOLD)

    motivo_full = None
    if FULL_REBUILD_ENV:
        motivo_full = 'HORUS_FULL_REBUILD=1'
    elif not gold_existe:
        motivo_full = 'tabla gold inexistente'
    elif estado_sv is None:
        motivo_full = 'silver sin tablas de control'
    elif not any(estado_gold.values()):
        motivo_full = 'sin historial de gold (primera corrida incremental)'
    elif fp_previo != fp_actual:
        motivo_full = 'cambio en maestros MDS o en VERSION_LOGICA'
    full = motivo_full is not None

    if full:
        print(f"[INFO] Modo: FULL REBUILD ({motivo_full}).")
        plan = {f: (None, []) for f in FUENTES}
    else:
        plan = determinar_pendientes(estado_sv, estado_gold)
        n_total = sum(len(r) + len(e) for r, e in plan.values())
        print("[INFO] Modo: INCREMENTAL.")
        for fuente, (rehacer, eliminar) in plan.items():
            print(f"   {fuente}: {len(rehacer)} lunes a rehacer, {len(eliminar)} a eliminar"
                  + (f" -> {[str(f) for f in rehacer][:12]}{'...' if len(rehacer) > 12 else ''}" if rehacer else ''))
        if n_total == 0:
            print("\n[OK] Nada pendiente. Gold ya esta al dia.")
            print(f"\n--- PROCESO GOLD FINALIZADO: {datetime.datetime.now()} ---")
            return

    if DRY_RUN:
        print("\n[DRY-RUN] HORUS_DRY_RUN=1: no se escribe nada.")
        return

    # 2. LEER SILVER (todo o solo los lunes afectados por fuente)
    print()
    df_copi = leer_silver_copi(plan['COPI'][0])
    df_snp  = leer_silver_snp(plan['SNP'][0])

    # 3. CONSTRUIR TABLA COMPARATIVA
    df_gold = construir_tabla_comparativa(df_copi, df_snp, df_eans, df_pdv)
    del df_copi, df_snp
    imprimir_resumen(df_gold)

    # 4. ESCRIBIR (una sola transaccion: gold + control)
    print(f"\n[INFO] Escribiendo en {SCHEMA}.{TABLA_GOLD}...")
    with engine.begin() as conn:
        if full:
            crear_tabla_desde_df(conn, df_gold, TABLA_GOLD)
            conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_SEMANAS}]"))
        else:
            for fuente, (rehacer, eliminar) in plan.items():
                afectadas = sorted(set(rehacer) | set(eliminar))
                borrar_fuente_fechas(conn, fuente, _a_datetime(afectadas))
                borrar_fuente_fechas(conn, fuente, list(afectadas), tabla=TABLA_SEMANAS)

        insertar(conn, df_gold, TABLA_GOLD)

        # Control: version de silver procesada por (Fuente, Fecha).
        # Si silver no tiene control (full por ese motivo), se registra sin
        # fecha_proceso_silver: la proxima corrida incremental los rehace.
        conteo = (df_gold.groupby(['Fuente', df_gold['Fecha'].dt.date]).size().to_dict()
                  if len(df_gold) else {})
        registros = []
        for fuente in FUENTES:
            fechas_sv = (estado_sv or {}).get(fuente, {})
            if full:
                candidatas = {f for (fu, f) in conteo if fu == fuente} | set(fechas_sv)
            else:
                candidatas = set(plan[fuente][0])
            for f in sorted(candidatas):
                registros.append({'fu': fuente, 'f': f, 'fp': fechas_sv.get(f),
                                  'n': int(conteo.get((fuente, f), 0))})
        if registros:
            conn.execute(text(f"""
                INSERT INTO [{SCHEMA}].[{TABLA_SEMANAS}] (Fuente, Fecha, fecha_proceso_silver, filas)
                VALUES (:fu, :f, :fp, :n)
            """), registros)

        escribir_control(conn, CLAVE_MAESTROS, fp_actual)

    print(f"[OK] PRC.{TABLA_GOLD} actualizada ({len(registros)} pares Fuente/Fecha registrados).")
    print(f"\n--- PROCESO GOLD FINALIZADO: {datetime.datetime.now()} ---")


if __name__ == "__main__":
    run_gold_comparativo()
